<a href="https://colab.research.google.com/github/vrajpsoni/Vraj-s_CS617_BostonPD/blob/main/Vraj's_CS617_BostonPD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [ ]:
import pandas as pd
import requests
import urllib.parse

In [ ]:
sql = '''
SELECT
    "INCIDENT_NUMBER",
    "OFFENSE_CODE_GROUP",
    "OFFENSE_DESCRIPTION",
    "DISTRICT",
    "SHOOTING",
    "OCCURRED_ON_DATE",
    "YEAR",
    "MONTH",
    "DAY_OF_WEEK",
    "HOUR",
    "Lat",
    "Long"
FROM "b973d8cb-eeb2-4e7e-99da-c92938efc9c0"
WHERE "OCCURRED_ON_DATE" IS NOT NULL
'''

In [ ]:
bpd_logs = "https://data.boston.gov/api/3/action/datastore_search_sql"

In [ ]:
url = bpd_logs+"?sql="+urllib.parse.quote(sql)
rows = requests.get( url ).json()

shootings = rows['result']['records']

In [ ]:
df = pd.DataFrame(rows['result']['records'])
df.head()

,INCIDENT_NUMBER,OFFENSE_CODE_GROUP,OFFENSE_DESCRIPTION,DISTRICT,SHOOTING,OCCURRED_ON_DATE,YEAR,MONTH,DAY_OF_WEEK,HOUR,Lat,Long
0,232007173,None,INVESTIGATE PERSON,B3,0,2023-01-27 22:44:00+00,2023,1,Friday,22,42.27166144598919,-71.09953381947697
1,232004454,None,VERBAL DISPUTE,B2,0,2023-01-17 20:21:00+00,2023,1,Tuesday,20,42.312596725464594,-71.09287521197788
2,232006290,None,INVESTIGATE PERSON,A1,0,2023-01-24 00:00:00+00,2023,1,Tuesday,0,42.36569984586877,-71.052891614826
3,232024939,None,INVESTIGATE PROPERTY,B3,0,2023-03-31 17:14:00+00,2023,3,Friday,17,42.29278842433874,-71.08851887959194
4,232006708,None,ASSAULT - AGGRAVATED,B2,0,2023-01-26 09:00:00+00,2023,1,Thursday,9,42.310269344309944,-71.0893099278548


In [ ]:
df.columns.tolist()

['INCIDENT_NUMBER',
 'OFFENSE_CODE_GROUP',
 'OFFENSE_DESCRIPTION',
 'DISTRICT',
 'SHOOTING',
 'OCCURRED_ON_DATE',
 'YEAR',
 'MONTH',
 'DAY_OF_WEEK',
 'HOUR',
 'Lat',
 'Long']

In [ ]:
df["OCCURRED_ON_DATE"] = pd.to_datetime(df["OCCURRED_ON_DATE"], errors="coerce")
df["HOUR"] = pd.to_numeric(df["HOUR"], errors="coerce")
df["Lat"] = pd.to_numeric(df["Lat"], errors="coerce")
df["Long"] = pd.to_numeric(df["Long"], errors="coerce")

df[["OCCURRED_ON_DATE", "HOUR", "Lat", "Long"]].head()

,OCCURRED_ON_DATE,HOUR,Lat,Long
0,2023-01-27 22:44:00+00:00,22,42.271661,-71.099534
1,2023-01-17 20:21:00+00:00,20,42.312597,-71.092875
2,2023-01-24 00:00:00+00:00,0,42.365700,-71.052892
3,2023-03-31 17:14:00+00:00,17,42.292788,-71.088519
4,2023-01-26 09:00:00+00:00,9,42.310269,-71.089310


In [ ]:
df = df.dropna(subset=["OCCURRED_ON_DATE", "HOUR", "Lat", "Long"])
df = df[(df["Lat"] != 0) & (df["Long"] != 0)]

df.shape

(30133, 12)

In [ ]:
df["is_night"] = df["HOUR"].apply(lambda h: 1 if (h >= 18 or h <= 2) else 0)

night_df = df[df["is_night"] == 1]

night_df[["HOUR"]].head()

,HOUR
0,22
1,20
2,0
9,18
11,21


In [ ]:
hour_summary = (
    night_df.groupby("HOUR")
    .size()
    .reset_index(name="crime_count")
    .sort_values("HOUR")
)

hour_summary

,HOUR,crime_count
0,0,2385
1,1,727
2,2,633
3,18,1846
4,19,1417
5,20,1406
6,21,1261
7,22,1122
8,23,771


In [ ]:
map_df = night_df[[
    "INCIDENT_NUMBER",
    "OFFENSE_CODE_GROUP",
    "OFFENSE_DESCRIPTION",
    "DISTRICT",
    "DAY_OF_WEEK",
    "HOUR",
    "Lat",
    "Long"
]].copy()

map_df.head()

,INCIDENT_NUMBER,OFFENSE_CODE_GROUP,OFFENSE_DESCRIPTION,DISTRICT,DAY_OF_WEEK,HOUR,Lat,Long
0,232007173,None,INVESTIGATE PERSON,B3,Friday,22,42.271661,-71.099534
1,232004454,None,VERBAL DISPUTE,B2,Tuesday,20,42.312597,-71.092875
2,232006290,None,INVESTIGATE PERSON,A1,Tuesday,0,42.365700,-71.052892
9,232012445,None,THREATS TO DO BODILY HARM,C6,Wednesday,18,42.326967,-71.061986
11,232013356,None,"MURDER, NON-NEGLIGENT MANSLAUGHTER",B3,Saturday,21,42.295982,-71.089760


In [ ]:
hour_summary.to_csv("night_crimes_by_hour.csv", index=False)
map_df.to_csv("night_crime_locations.csv", index=False)

In [ ]:
shootings

[{'DISTRICT': 'B3',
  'OCCURRED_ON_DATE': '2023-02-18 21:43:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'MURDER, NON-NEGLIGENT MANSLAUGHTER'},
 {'DISTRICT': 'D4',
  'OCCURRED_ON_DATE': '2023-05-24 18:49:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'BALLISTICS EVIDENCE/FOUND'},
 {'DISTRICT': 'D4',
  'OCCURRED_ON_DATE': '2023-10-27 12:50:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'ASSAULT - AGGRAVATED'},
 {'DISTRICT': 'B3',
  'OCCURRED_ON_DATE': '2023-01-07 17:51:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'INVESTIGATE PROPERTY'},
 {'DISTRICT': 'B2',
  'OCCURRED_ON_DATE': '2023-01-30 17:00:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'ASSAULT - AGGRAVATED'},
 {'DISTRICT': 'C11',
  'OCCURRED_ON_DATE': '2023-10-28 20:56:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'BALLISTICS EVIDENCE/FOUND'},
 {'DISTRICT': 'C11',
  'OCCURRED_ON_DATE': '2024-08-12 18:40:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'MURDER, NON-NEGLIGENT MANSLAUGHTER'},
 